# R3-N1 Stage 1: Nested Grouped Cross-Validation Calibration Experiments

**Project:** EchoProof  
**Task:** R3-N1 Stage 1 — Nested Grouped Cross-Validation for Calibration  
**Model Under Calibration:** R3-M1 Protocol B BiGRU  
**Source Run:** `runs/r3/r3_m1_20261107T144037Z/`  
**Dataset Role:** Calibration Role Only (960 recordings, 12 speaker groups)  
**Runtime Requirement:** Standard Google Colab CPU Runtime (GPU not required)  

---

## Executive Summary & Scientific Scope

This notebook implements **Nested Grouped Cross-Validation** to rigorously evaluate and compare five recording-level probability calibration methods:

1. **Scalar Temperature Scaling (baseline):** 1-parameter log-odds scaling ($T > 0$).
2. **Platt Scaling:** 2-parameter logistic calibration ($a \cdot z + b$) with slope strictly constrained to $a > 0$ to guarantee score ranking preservation.
3. **Standard Isotonic Regression:** Non-parametric monotonic step mapping via Pair Adjacent Violators Algorithm (PAVA).
4. **Regularized Isotonic Regression:** Parametric shrinkage towards temperature scaling ($p = (1 - \lambda) p_{\text{iso}} + \lambda p_{\text{temp}}$), where shrinkage weight $\lambda \in [0, 1]$ is selected strictly within inner cross-validation folds.
5. **Smoothed Isotonic Regression:** Shape-preserving $C^1$-smooth cubic Hermite spline interpolation (PCHIP) applied to $K$ quantile knots of the isotonic mapping, where knot count $K \in [5, 30]$ is selected strictly within inner cross-validation folds.

### Why Nested Cross-Validation is Necessary
In standard flat cross-validation, selecting hyperparameters (such as $\lambda$ or $K$) on the same cross-validation splits used to report out-of-fold performance introduces optimization bias (information leakage from validation folds into hyperparameter selection). Nested cross-validation enforces strict structural separation:
- **Outer CV (4 Folds):** Partitions the 12 speaker groups into training (9 groups, 720 recordings) and validation (3 groups, 240 recordings) sets. Provides nested grouped OOF estimates on held-out speaker groups; it does not establish an unbiased generalization guarantee.
- **Inner CV (3 Folds):** Operates **strictly on the outer training partition** (9 groups), partitioning into 6 training groups and 3 validation groups to select $\lambda$ and $K$.
- **Outer-fold isolation:** The outer validation partition is never seen during inner hyperparameter tuning or model fitting.

### Strict Boundaries & Isolation Rules
- **Calibration Role Partition Only:** Strictly uses `predictions_recordings_calibration.parquet` (960 recordings: 240 genuine, 240 Melo, 240 VITS, 240 XTTS).
- **Zero Operating or Test Data Access:** Operating and test partition recordings and labels are strictly excluded.
- **Zero Chatterbox Access:** No Chatterbox audio, recordings, or predictions are accessed.
- **Zero BiGRU Modification:** Underlying model weights are untouched; all methods are post-processing calibrations.
- **Speaker-group separation:** All splits are grouped strictly by `group_id` across both outer and inner folds.

## Section 1: Google Drive & Project Setup

Mounts Google Drive to access persistent project storage at `/content/drive/MyDrive/PAI&AIES Project`, configures the project root and prediction directory, and sets Python search paths to import the EchoProof package.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import hashlib
import inspect
import json
import math
import os
from pathlib import Path
import platform
import sys
import time
from datetime import datetime, timezone

# 1. Canonical Project Root Configuration
ROOT = Path('/content/drive/MyDrive/PAI&AIES Project')
if not (ROOT / 'src/echoproof').is_dir():
    alt_roots = [
        Path('/content/drive/My Drive/PAI&AIES Project'),
        Path(os.environ.get('ECHOPROOF_ROOT', '')),
        Path.cwd(),
        Path.cwd().parent,
    ]
    for cand in alt_roots:
        if cand and (cand / 'src/echoproof').is_dir():
            ROOT = cand.resolve()
            break

print(f"[Project Root] {ROOT}")
assert (ROOT / 'src/echoproof').is_dir(), f"Cannot locate EchoProof source directory under: {ROOT}"

# 2. Source Prediction Directory & Artifacts
R3_M1_RUN_ID = "r3_m1_20261107T144037Z"
PRED_DIR = ROOT / 'runs/r3' / R3_M1_RUN_ID
CALIBRATION_PRED_FILE = PRED_DIR / 'predictions_recordings_calibration.parquet'

print(f"[Prediction Dir]   : {PRED_DIR}")
print(f"[Calibration File] : {CALIBRATION_PRED_FILE}")
assert PRED_DIR.is_dir(), f"Source prediction run directory not found: {PRED_DIR}"
assert CALIBRATION_PRED_FILE.is_file(), f"Calibration predictions parquet not found: {CALIBRATION_PRED_FILE}"

# 3. Configure Python Environment & Search Paths
sys.dont_write_bytecode = True
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
os.environ['PYTHONPATH'] = str(ROOT / 'src')

# 4. Dependency Imports & Versions
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import scipy
from scipy.optimize import minimize, minimize_scalar
from scipy.interpolate import PchipInterpolator
import sklearn
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import GroupKFold

from echoproof.evaluation.metrics import (
    _compute_roc_auc,
    interpolated_eer,
)

print("\n[Environment Audit]")
print(f"  Python       : {platform.python_version()}")
print(f"  Platform     : {platform.platform()}")
print(f"  NumPy        : {np.__version__}")
print(f"  SciPy        : {scipy.__version__}")
print(f"  Pandas       : {pd.__version__}")
print(f"  PyArrow      : {pa.__version__}")
print(f"  Scikit-Learn : {sklearn.__version__}")
print(f"  Matplotlib   : {matplotlib.__version__}")

## Section 2: Load and Validate Calibration Dataset

Loads `predictions_recordings_calibration.parquet` and performs strict contract validation:
- Validates presence of required columns: `recording_id`, `group_id`, `label`, `component`, `validation_role`, `raw_score`, `chunk_count`.
- Verifies zero null or missing values.
- Asserts uniqueness of `recording_id`.
- Validates binary labels ($y \in \{0, 1\}$) and finite probabilities ($s \in [0, 1]$).
- Asserts that all rows belong strictly to the `calibration` validation role.
- Validates exact cohort counts: 960 total recordings (240 genuine, 240 Melo, 240 VITS, 240 XTTS).
- Inspects and verifies the 12 speaker groups (`group_id`) of 80 recordings each.
- Confirms zero Chatterbox recordings are present.
- Computes SHA-256 of the source prediction file before proceeding.

In [ ]:
def compute_sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(1048576), b''):
            h.update(chunk)
    return h.hexdigest()

source_pred_sha256 = compute_sha256(CALIBRATION_PRED_FILE)
source_pred_size = CALIBRATION_PRED_FILE.stat().st_size
EXPECTED_SHA256 = "2a1b6aa6b8bcbcca578a047ea321210b755a5de5ff815e1705a7b1d9374d26c3"

print("[Source Artifact Verification]")
print(f"  Path    : {CALIBRATION_PRED_FILE.relative_to(ROOT)}")
print(f"  SHA-256 : {source_pred_sha256}")
print(f"  Bytes   : {source_pred_size}")
assert source_pred_sha256 == EXPECTED_SHA256, f"Source SHA-256 mismatch! Got: {source_pred_sha256}"

# Read Calibration Predictions
cal_df = pq.read_table(CALIBRATION_PRED_FILE).to_pandas()
print(f"\n[Data Loaded] Shape: {cal_df.shape}")

REQUIRED_COLS = [
    'recording_id', 'group_id', 'label', 'component',
    'validation_role', 'raw_score', 'chunk_count'
]
missing_cols = [c for c in REQUIRED_COLS if c not in cal_df.columns]
assert not missing_cols, f"Missing required columns in prediction parquet: {missing_cols}"

# Check for nulls
null_counts = cal_df[REQUIRED_COLS].isna().sum()
assert null_counts.sum() == 0, f"Found missing values in calibration data:\n{null_counts[null_counts > 0]}"

# Unique recording IDs
assert cal_df['recording_id'].nunique() == len(cal_df), "Recording IDs are not unique!"

# Binary labels (0 = Genuine, 1 = TTS)
unique_labels = sorted(cal_df['label'].unique().tolist())
assert unique_labels == [0, 1], f"Labels must be strictly binary {{0, 1}}, got: {unique_labels}"

# Finite probabilities in [0, 1]
raw_scores_arr = cal_df['raw_score'].values
assert np.all(np.isfinite(raw_scores_arr)), "Found non-finite raw scores!"
assert np.all(raw_scores_arr >= 0.0) and np.all(raw_scores_arr <= 1.0), "Raw scores outside [0, 1]!"

# Validation role must be 'calibration'
roles = cal_df['validation_role'].unique().tolist()
assert roles == ['calibration'], f"Unexpected validation role(s): {roles}"

# Exact Reference Count Validation
TOTAL_EXPECTED = 960
COMPONENT_EXPECTED = {
    'genuine': 240,
    'melo': 240,
    'vits': 240,
    'xtts': 240,
}
assert len(cal_df) == TOTAL_EXPECTED, f"Expected {TOTAL_EXPECTED} total recordings, got {len(cal_df)}"

comp_counts = cal_df['component'].value_counts().to_dict()
for comp_name, expected_count in COMPONENT_EXPECTED.items():
    actual_count = comp_counts.get(comp_name, 0)
    assert actual_count == expected_count, (
        f"Component '{comp_name}' count mismatch: expected {expected_count}, got {actual_count}"
    )

label_counts = cal_df['label'].value_counts().to_dict()
assert label_counts.get(0, 0) == 240, f"Expected 240 genuine recordings (label 0), got {label_counts.get(0)}"
assert label_counts.get(1, 0) == 720, f"Expected 720 TTS recordings (label 1), got {label_counts.get(1)}"

EXPECTED_COMPONENT_LABELS = {'genuine': 0, 'melo': 1, 'vits': 1, 'xtts': 1}
for comp_name, expected_label in EXPECTED_COMPONENT_LABELS.items():
    observed_labels = sorted(cal_df.loc[cal_df['component'] == comp_name, 'label'].unique().tolist())
    assert observed_labels == [expected_label], f"{comp_name} must map to label {expected_label}, got {observed_labels}"

# Speaker/Group distribution audit
unique_groups = cal_df['group_id'].nunique()
group_counts = cal_df['group_id'].value_counts().to_dict()
assert unique_groups == 12, f"Expected 12 unique speaker groups, got {unique_groups}"
assert all(cnt == 80 for cnt in group_counts.values()), "All groups must have exactly 80 recordings each"

# Zero Chatterbox check
assert 'chatterbox' not in comp_counts, "Security violation: Chatterbox data detected in calibration set!"

print("\n[Calibration Dataset Verification: PASS]")
print(f"  Total Recordings   : {len(cal_df)}")
print(f"  Class Distribution : Genuine (0) = {label_counts[0]}, TTS (1) = {label_counts[1]}")
print(f"  Speaker Groups     : {unique_groups} groups (80 recordings/group, zero cross-group contamination)")
print(f"  Component Breakdown:")
for comp, cnt in sorted(comp_counts.items()):
    print(f"    - {comp:<10} : {cnt}")

## Section 3: Authoritative Metric Functions & Reliability Binning

Defines the core evaluation metrics and binning procedures for calibration evaluation:
1. **Binary Cross-Entropy (Log Loss / BCE):**
   $$\mathcal{L}_{\text{BCE}} = -\frac{1}{N}\sum_{i=1}^N \left[y_i \ln(s_{i,\epsilon}) + (1 - y_i)\ln(1 - s_{i,\epsilon})\right], \quad \epsilon = 10^{-7}$$
2. **Brier Score:**
   $$\text{BS} = \frac{1}{N}\sum_{i=1}^N (s_i - y_i)^2$$
3. **Equal-Width Expected Calibration Error (ECE):**
   Fixed uniform binning ($B=10$ bins: $[0, 0.1), \dots, [0.9, 1.0]$):
   $$\text{ECE}_{\text{width}} = \sum_{b=1}^B \frac{|B_b|}{N} |\text{acc}(B_b) - \text{conf}(B_b)|$$
   *Empty bin handling:* Empty bins ($|B_b| = 0$) are explicitly tracked and excluded from calibration accuracy calculations. They are not treated as evidence of perfect calibration.
4. **Equal-Mass Expected Calibration Error (ECE):**
   Quantile binning ($B=10$ bins, each containing $\sim 10\%$ of recordings):
   *Tied boundary handling:* Deduplicates identical quantile boundaries to ensure well-defined intervals.
5. **Non-Decreasing Verification:** Verifies $s_i > s_j \implies p(s_i) \ge p(s_j)$ over predictions and dense grids.

In [ ]:
from echoproof.evaluation.calibrate import DEFAULT_EPSILON, apply_temperature
from scipy.special import expit

def logit(p: np.ndarray, eps: float = DEFAULT_EPSILON) -> np.ndarray:
    p_clip = np.clip(p, eps, 1.0 - eps)
    return np.log(p_clip / (1.0 - p_clip))

def sigmoid(z: np.ndarray) -> np.ndarray:
    return expit(np.asarray(z, dtype=np.float64))

def fit_temperature(labels, scores):
    """Notebook-local temperature fit with stable softplus cross-entropy."""
    labels = np.asarray(labels, dtype=np.int64)
    scores = np.asarray(scores, dtype=np.float64)
    assert set(np.unique(labels)) == {0, 1}, "Temperature fit requires both classes"
    logits = logit(scores)
    signed_labels = 1.0 - 2.0 * labels
    result = minimize_scalar(
        lambda temperature: float(np.mean(np.logaddexp(0.0, signed_labels * logits / temperature))),
        bounds=(0.01, 50.0), method="bounded"
    )
    assert result.success and result.x > 0, f"Temperature optimization failed: {result.message}"
    return float(result.x)

def compute_bce_loss(labels: np.ndarray, scores: np.ndarray, eps: float = DEFAULT_EPSILON) -> float:
    s_clipped = np.clip(scores, eps, 1.0 - eps)
    return float(-np.mean(labels * np.log(s_clipped) + (1.0 - labels) * np.log(1.0 - s_clipped)))

def compute_brier_score(labels: np.ndarray, scores: np.ndarray) -> float:
    return float(np.mean((scores - labels) ** 2))

def compute_equal_width_ece(labels: np.ndarray, scores: np.ndarray, n_bins: int = 10) -> tuple[float, list[dict], int]:
    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    total = len(labels)
    weighted_err = 0.0
    bins_info = []
    
    for i in range(n_bins):
        low, high = bin_edges[i], bin_edges[i + 1]
        mask = (scores >= low) & (scores <= high) if i == n_bins - 1 else (scores >= low) & (scores < high)
        cnt = int(np.sum(mask))
        if cnt > 0:
            conf = float(np.mean(scores[mask]))
            acc = float(np.mean(labels[mask]))
            err = abs(acc - conf)
            weighted_err += (cnt / total) * err
            bins_info.append({
                'bin_idx': i,
                'bin_type': 'equal_width',
                'bin_range': f"[{low:.1f}, {high:.1f}]" if i == n_bins - 1 else f"[{low:.1f}, {high:.1f})",
                'low': float(low),
                'high': float(high),
                'count': cnt,
                'confidence': conf,
                'accuracy': acc,
                'calibration_error': err,
                'is_empty': False,
            })
        else:
            bins_info.append({
                'bin_idx': i,
                'bin_type': 'equal_width',
                'bin_range': f"[{low:.1f}, {high:.1f}]" if i == n_bins - 1 else f"[{low:.1f}, {high:.1f})",
                'low': float(low),
                'high': float(high),
                'count': 0,
                'confidence': None,
                'accuracy': None,
                'calibration_error': None,
                'is_empty': True,
            })
    ece = float(weighted_err)
    non_empty_bins = sum(1 for b in bins_info if not b['is_empty'])
    return ece, bins_info, non_empty_bins

def compute_equal_mass_ece(labels: np.ndarray, scores: np.ndarray, n_bins: int = 10) -> tuple[float, list[dict], int]:
    total = len(labels)
    quantiles = np.linspace(0.0, 1.0, n_bins + 1)
    raw_edges = np.quantile(scores, quantiles)
    unique_edges = np.unique(raw_edges)
    actual_bins = len(unique_edges) - 1
    
    weighted_err = 0.0
    bins_info = []
    
    for i in range(actual_bins):
        low, high = unique_edges[i], unique_edges[i + 1]
        mask = (scores >= low) & (scores <= high) if i == actual_bins - 1 else (scores >= low) & (scores < high)
        cnt = int(np.sum(mask))
        if cnt > 0:
            conf = float(np.mean(scores[mask]))
            acc = float(np.mean(labels[mask]))
            err = abs(acc - conf)
            weighted_err += (cnt / total) * err
            bins_info.append({
                'bin_idx': i,
                'bin_type': 'equal_mass',
                'bin_range': f"[{low:.4f}, {high:.4f}]" if i == actual_bins - 1 else f"[{low:.4f}, {high:.4f})",
                'low': float(low),
                'high': float(high),
                'count': cnt,
                'confidence': conf,
                'accuracy': acc,
                'calibration_error': err,
                'is_empty': False,
            })
        else:
            bins_info.append({
                'bin_idx': i,
                'bin_type': 'equal_mass',
                'bin_range': f"[{low:.4f}, {high:.4f}]" if i == actual_bins - 1 else f"[{low:.4f}, {high:.4f})",
                'low': float(low),
                'high': float(high),
                'count': 0,
                'confidence': None,
                'accuracy': None,
                'calibration_error': None,
                'is_empty': True,
            })
    ece = float(weighted_err)
    non_empty_bins = sum(1 for b in bins_info if not b['is_empty'])
    return ece, bins_info, non_empty_bins

def check_non_decreasing(raw: np.ndarray, cal: np.ndarray) -> bool:
    sorted_idx = np.argsort(raw)
    diffs = np.diff(cal[sorted_idx])
    return bool(np.all(diffs >= -1e-12))

def calculate_metrics(labels, scores):
    labels = np.asarray(labels, dtype=np.int64)
    scores = np.asarray(scores, dtype=np.float64)
    assert set(np.unique(labels)) == {0, 1}, "Log Loss, ROC-AUC, and EER require both classes"
    assert np.all(np.isfinite(scores)) and np.all((scores >= 0.0) & (scores <= 1.0)), "Metrics require finite probabilities in [0,1]"
    ece_width, _, _ = compute_equal_width_ece(labels, scores, n_bins=10)
    ece_mass, _, _ = compute_equal_mass_ece(labels, scores, n_bins=10)
    return {"log_loss": compute_bce_loss(labels, scores), "brier_score": compute_brier_score(labels, scores),
            "equal_width_ece": ece_width, "equal_mass_ece": ece_mass,
            "roc_auc": _compute_roc_auc(labels, scores), "eer": interpolated_eer(labels, scores)}

y_true = cal_df['label'].values.astype(np.int64)
s_raw = cal_df['raw_score'].values.astype(np.float64)
groups_arr = cal_df['group_id'].values

print("Authoritative metrics and binning utilities loaded.")

## Section 4: Calibration Method Definitions & Fitting Utilities

Defines the model fitting and transformation procedures for each method:
1. **Temperature Scaling:** Fits positive scalar $T > 0$ via bounded minimization on log loss.
2. **Platt Scaling:** Fits slope $a > 0$ and intercept $b$ via L-BFGS-B bounded optimization on raw logit features ($z = \text{logit}(s)$).
3. **Standard Isotonic Regression:** Non-parametric step function via PAVA (`IsotonicRegression(y_min=0, y_max=1, out_of_bounds='clip')`).
4. **Regularized Isotonic Regression:** Shrinkage blend $(1 - \lambda) p_{\text{iso}} + \lambda p_{\text{temp}}$.
5. **Smoothed Isotonic Regression:** PCHIP interpolation through fitted isotonic predictions at selected quantile knots. Input knots include 0 and 1; their output values are predictions from the fitted isotonic model, not forced endpoint anchors.

### Monotonicity Verification for PCHIP
PCHIP is mathematically proven to strictly preserve monotonicity across non-decreasing knots (Fritsch & Carlson, 1980). Each fitted map is checked separately on a dense input grid within its fold; inner PCHIP fits are checked during tuning.

In [ ]:
def fit_platt_model(s_train: np.ndarray, y_train: np.ndarray) -> tuple[float, float]:
    z_tr = logit(s_train)
    t_init = fit_temperature(y_train.tolist(), s_train.tolist())
    
    def platt_loss(params: tuple[float, float]) -> float:
        a, b = params
        z_scaled = a * z_tr + b
        margin = (1.0 - 2.0 * y_train) * z_scaled
        loss = np.logaddexp(0.0, margin)
        return float(np.mean(loss))
        
    res = minimize(platt_loss, x0=[1.0 / t_init, 0.0], bounds=[(1e-4, 50.0), (-20.0, 20.0)], method='L-BFGS-B')
    assert res.success, f"Platt optimization failed: {res.message}"
    a_val, b_val = float(res.x[0]), float(res.x[1])
    assert a_val > 0.0, f"Platt slope must be positive, got {a_val}"
    return a_val, b_val

def apply_platt_model(s_eval: np.ndarray, a: float, b: float) -> np.ndarray:
    z_eval = logit(s_eval)
    return sigmoid(a * z_eval + b)

def fit_smoothed_pchip(s_train: np.ndarray, iso_model: IsotonicRegression, k_knots: int) -> PchipInterpolator:
    q_knots = np.linspace(0.0, 1.0, k_knots)
    x_k = np.unique(np.concatenate([[0.0], np.quantile(s_train, q_knots), [1.0]]))
    y_k = iso_model.predict(x_k)
    assert np.all(np.diff(y_k) >= -1e-12), "Isotonic knot predictions must be non-decreasing"
    return PchipInterpolator(x_k, y_k)

print("Calibration fitting utilities loaded; monotonicity is checked for each fold-specific fitted map.")


## Section 5: Nested Grouped Cross-Validation Engine

Executes nested grouped cross-validation across all 960 calibration recordings:
- **Outer Cross-Validation (4 Folds):**
  - Grouped by `group_id` (12 groups $\implies$ 3 validation groups / 240 recordings per fold, 9 training groups / 720 recordings).
  - Evaluates out-of-fold generalization.
- **Inner Cross-Validation (3 Folds):**
  - Applied strictly to the 9 training groups of each outer fold (each inner fold holds out 3 groups = 240 recordings, trains on 6 groups = 480 recordings).
  - Selects optimal $\lambda \in [0.0, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]$ for Regularized Isotonic.
  - Selects optimal $K \in [5, 10, 15, 20, 30]$ knots for Smoothed Isotonic.
- **Outer Fold Fitting & Prediction:**
  - Fits Temperature Scaling, Platt Scaling, Standard Isotonic, Regularized Isotonic (using inner $\lambda^*$), and Smoothed Isotonic (using inner $K^*$) on the 720 outer training recordings.
  - Generates predictions on the untouched 240 outer validation recordings.
  - Assembles one held-out prediction per recording, records complete outer and inner group manifests, and checks fold-specific monotonicity.

In [ ]:
outer_gkf = GroupKFold(n_splits=4)
inner_gkf = GroupKFold(n_splits=3)
lambda_grid = [0.0, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
knot_candidates = [5, 10, 15, 20, 30]
method_keys = ["raw", "temperature", "platt", "isotonic", "regularized_isotonic", "smoothed_isotonic"]
nested_oof_preds = {"raw": np.copy(s_raw), **{key: np.full_like(s_raw, np.nan, dtype=np.float64) for key in method_keys if key != "raw"}}
oof_assignment_counts = np.zeros(len(s_raw), dtype=np.int16)
outer_fold_assignment = np.full(len(s_raw), -1, dtype=np.int16)
nested_fold_records, inner_cv_selection_curves, fold_manifest = [], [], []
outer_fold_predictions = []
dense_grid = np.linspace(0.0, 1.0, 1001)

for outer_fold, (outer_train_idx, outer_val_idx) in enumerate(outer_gkf.split(s_raw, y_true, groups=groups_arr), start=1):
    y_train, y_val = y_true[outer_train_idx], y_true[outer_val_idx]
    s_train, s_val = s_raw[outer_train_idx], s_raw[outer_val_idx]
    train_groups = sorted(str(g) for g in set(groups_arr[outer_train_idx]))
    val_groups = sorted(str(g) for g in set(groups_arr[outer_val_idx]))
    assert set(train_groups).isdisjoint(val_groups), f"Outer group leakage in fold {outer_fold}"
    assert set(np.unique(y_train)) == {0, 1} and set(np.unique(y_val)) == {0, 1}, f"Both classes required in outer fold {outer_fold}"
    assert np.all(oof_assignment_counts[outer_val_idx] == 0), f"Duplicate OOF assignment in outer fold {outer_fold}"
    manifest_row = {
        "outer_fold": outer_fold,
        "outer_train_groups": train_groups,
        "outer_validation_groups": val_groups,
        "outer_train_recording_ids": [str(x) for x in cal_df.iloc[outer_train_idx]["recording_id"].tolist()],
        "outer_validation_recording_ids": [str(x) for x in cal_df.iloc[outer_val_idx]["recording_id"].tolist()],
        "inner_folds": [],
    }
    fold_manifest.append(manifest_row)
    inner_reg_losses = {value: [] for value in lambda_grid}
    inner_smooth_losses = {value: [] for value in knot_candidates}

    for inner_fold, (inner_train_local, inner_val_local) in enumerate(inner_gkf.split(s_train, y_train, groups=groups_arr[outer_train_idx]), start=1):
        inner_train_idx = outer_train_idx[inner_train_local]
        inner_val_idx = outer_train_idx[inner_val_local]
        y_inner_train, y_inner_val = y_true[inner_train_idx], y_true[inner_val_idx]
        s_inner_train, s_inner_val = s_raw[inner_train_idx], s_raw[inner_val_idx]
        inner_train_groups = sorted(str(g) for g in set(groups_arr[inner_train_idx]))
        inner_val_groups = sorted(str(g) for g in set(groups_arr[inner_val_idx]))
        assert set(inner_train_groups).isdisjoint(inner_val_groups), f"Inner group leakage in outer {outer_fold}, inner {inner_fold}"
        assert set(inner_train_groups + inner_val_groups).issubset(train_groups), "Inner split includes a group outside outer training"
        assert set(np.unique(y_inner_train)) == {0, 1} and set(np.unique(y_inner_val)) == {0, 1}, "Both classes required in inner fold"
        manifest_row["inner_folds"].append({
            "inner_fold": inner_fold,
            "train_groups": inner_train_groups,
            "validation_groups": inner_val_groups,
            "train_recording_ids": [str(x) for x in cal_df.iloc[inner_train_idx]["recording_id"].tolist()],
            "validation_recording_ids": [str(x) for x in cal_df.iloc[inner_val_idx]["recording_id"].tolist()],
        })
        temperature_inner = fit_temperature(y_inner_train, s_inner_train)
        p_temp_inner = np.asarray(apply_temperature(s_inner_val.tolist(), temperature_inner))
        iso_inner = IsotonicRegression(y_min=0.0, y_max=1.0, out_of_bounds="clip").fit(s_inner_train, y_inner_train)
        p_iso_inner = np.clip(iso_inner.predict(s_inner_val), DEFAULT_EPSILON, 1.0 - DEFAULT_EPSILON)
        for lam in lambda_grid:
            inner_reg_losses[lam].append(compute_bce_loss(y_inner_val, (1.0 - lam) * p_iso_inner + lam * p_temp_inner))
        for knots in knot_candidates:
            pchip_inner = fit_smoothed_pchip(s_inner_train, iso_inner, k_knots=knots)
            inner_curve = np.asarray(pchip_inner(dense_grid), dtype=np.float64)
            assert np.all(np.isfinite(inner_curve)) and np.all((inner_curve >= 0) & (inner_curve <= 1)), "Invalid inner PCHIP grid probabilities"
            assert np.all(np.diff(inner_curve) >= -1e-12), f"Inner PCHIP not monotonic: outer {outer_fold}, inner {inner_fold}, K={knots}"
            inner_smooth_losses[knots].append(compute_bce_loss(y_inner_val, np.clip(pchip_inner(s_inner_val), DEFAULT_EPSILON, 1.0 - DEFAULT_EPSILON)))

    mean_reg_loss = {key: float(np.mean(values)) for key, values in inner_reg_losses.items()}
    mean_smooth_loss = {key: float(np.mean(values)) for key, values in inner_smooth_losses.items()}
    best_lambda = float(min(mean_reg_loss, key=mean_reg_loss.get))
    best_knots = int(min(mean_smooth_loss, key=mean_smooth_loss.get))
    inner_cv_selection_curves.append({"outer_fold": outer_fold, "reg_curve": mean_reg_loss, "smooth_curve": mean_smooth_loss, "selected_lambda": best_lambda, "selected_k": best_knots})

    temperature = fit_temperature(y_train, s_train)
    p_temp = np.asarray(apply_temperature(s_val.tolist(), temperature))
    platt_a, platt_b = fit_platt_model(s_train, y_train)
    p_platt = np.asarray(apply_platt_model(s_val, platt_a, platt_b))
    iso_outer = IsotonicRegression(y_min=0.0, y_max=1.0, out_of_bounds="clip").fit(s_train, y_train)
    p_iso = np.clip(iso_outer.predict(s_val), DEFAULT_EPSILON, 1.0 - DEFAULT_EPSILON)
    p_reg = (1.0 - best_lambda) * p_iso + best_lambda * p_temp
    pchip_outer = fit_smoothed_pchip(s_train, iso_outer, k_knots=best_knots)
    p_smooth = np.clip(pchip_outer(s_val), DEFAULT_EPSILON, 1.0 - DEFAULT_EPSILON)
    predictions = {"raw": s_val.copy(), "temperature": p_temp, "platt": p_platt, "isotonic": p_iso, "regularized_isotonic": p_reg, "smoothed_isotonic": p_smooth}

    grid_temp = np.asarray(apply_temperature(dense_grid.tolist(), temperature))
    grid_platt = np.asarray(apply_platt_model(dense_grid, platt_a, platt_b))
    grid_iso = np.clip(iso_outer.predict(dense_grid), DEFAULT_EPSILON, 1.0 - DEFAULT_EPSILON)
    grid_reg = (1.0 - best_lambda) * grid_iso + best_lambda * grid_temp
    grid_smooth = np.asarray(pchip_outer(dense_grid))
    monotonicity = {
        "raw_identity": bool(np.all(np.diff(dense_grid) >= 0)),
        "temperature": bool(np.all(np.diff(grid_temp) >= -1e-12)),
        "platt": bool(np.all(np.diff(grid_platt) >= -1e-12)),
        "isotonic": bool(np.all(np.diff(grid_iso) >= -1e-12)),
        "regularized_isotonic": bool(np.all(np.diff(grid_reg) >= -1e-12)),
        "smoothed_isotonic": bool(np.all(np.diff(grid_smooth) >= -1e-12)),
    }
    assert all(monotonicity.values()), f"Non-monotonic map in outer fold {outer_fold}: {monotonicity}"

    for method, values in predictions.items():
        values = np.asarray(values, dtype=np.float64)
        assert values.shape == s_val.shape, f"Wrong prediction shape for {method}, outer fold {outer_fold}"
        assert np.all(np.isfinite(values)), f"Non-finite prediction for {method}, outer fold {outer_fold}"
        assert np.all((values >= 0.0) & (values <= 1.0)), f"Probability outside [0,1] for {method}, outer fold {outer_fold}"
        if method != "raw":
            nested_oof_preds[method][outer_val_idx] = values
    oof_assignment_counts[outer_val_idx] += 1
    outer_fold_assignment[outer_val_idx] = outer_fold
    outer_fold_predictions.append({"outer_fold": outer_fold, "predictions": predictions})
    metrics_by_method = {method: calculate_metrics(y_val, scores) for method, scores in predictions.items()}
    nested_fold_records.append({
        "outer_fold": outer_fold,
        "train_groups": train_groups,
        "validation_groups": val_groups,
        "selected_lambda": best_lambda,
        "selected_k": best_knots,
        "fitted_T": float(temperature),
        "fitted_platt_a": float(platt_a),
        "fitted_platt_b": float(platt_b),
        "monotonicity_by_method": monotonicity,
        "metrics_by_method": metrics_by_method,
    })

assert np.all(oof_assignment_counts == 1), f"Each recording must receive exactly one OOF assignment: {np.unique(oof_assignment_counts, return_counts=True)}"
assert np.all(outer_fold_assignment >= 1), "Missing outer fold assignment"
assert np.all(outer_fold_assignment <= 4), "Invalid outer fold assignment"
for method, values in nested_oof_preds.items():
    assert np.all(np.isfinite(values)), f"Missing/non-finite OOF predictions for {method}"
    assert np.all((values >= 0.0) & (values <= 1.0)), f"OOF probabilities outside [0,1] for {method}"
assert all(set(row["outer_train_groups"]).isdisjoint(row["outer_validation_groups"]) for row in fold_manifest)
assert all(set(inner["train_groups"]).isdisjoint(inner["validation_groups"]) for row in fold_manifest for inner in row["inner_folds"])
print("Nested grouped OOF assignment, group-separation, and probability checks passed.")


## Section 6: Nested Out-of-Fold Performance Evaluation & Metrics

Reports nested grouped out-of-fold (OOF) estimates on held-out speaker groups for all five methods and the uncalibrated baseline:
- **Pooled Out-of-Fold Metrics (All 960 Recordings):** Log Loss, Brier Score, Equal-Width ECE (10 bins), Equal-Mass ECE (10 bins), ROC-AUC, and Equal Error Rate (EER).
- **Per-Outer-Fold Breakdown:** Shows stability across speaker splits.
- **Mean $\pm$ Standard Deviation across outer folds.**

> [!IMPORTANT]
> **Scientific Limitations of Grouped Cross-Validation:**
> Only 12 speaker groups are available, with 3 held out per outer fold. Fold estimates therefore have limited speaker support and do not establish an unbiased generalization guarantee. Pooled ROC-AUC and EER are supplementary because the pooled predictions come from different fold-specific calibrators.

In [ ]:
method_display_names = {
    "raw": "Raw identity mapping",
    "temperature": "1. Temperature Scaling",
    "platt": "2. Platt Scaling",
    "isotonic": "3. Standard Isotonic",
    "regularized_isotonic": "4. Regularized Isotonic",
    "smoothed_isotonic": "5. Smoothed Isotonic",
}
oof_bins_data = {}
oof_summary_rows = []
per_fold_metric_rows = []
metric_names = ["log_loss", "brier_score", "equal_width_ece", "equal_mass_ece", "roc_auc", "eer"]
for method, display_name in method_display_names.items():
    pooled_scores = nested_oof_preds[method]
    pooled_metrics = calculate_metrics(y_true, pooled_scores)
    bins_width = compute_equal_width_ece(y_true, pooled_scores, n_bins=10)[1]
    bins_mass = compute_equal_mass_ece(y_true, pooled_scores, n_bins=10)[1]
    oof_bins_data[method] = (bins_width, bins_mass)
    oof_summary_rows.append({"method": method, "Method": display_name, **pooled_metrics,
                             "roc_auc_eer_interpretation": "Supplementary pooled metric across fold-specific calibrators"})
    for fold_record, fold_predictions in zip(nested_fold_records, outer_fold_predictions):
        assert fold_record["outer_fold"] == fold_predictions["outer_fold"]
        fold_id = fold_record["outer_fold"]
        fold_mask = outer_fold_assignment == fold_id
        fold_metrics = calculate_metrics(y_true[fold_mask], fold_predictions["predictions"][method])
        per_fold_metric_rows.append({"outer_fold": fold_id, "method": method, "Method": display_name,
                                     "raw_identity_monotonic": fold_record["monotonicity_by_method"]["raw_identity"],
                                     "fitted_map_monotonic": fold_record["monotonicity_by_method"]["raw_identity"] if method == "raw" else fold_record["monotonicity_by_method"][method],
                                     **fold_metrics, "selected_lambda": fold_record["selected_lambda"],
                                     "selected_k": fold_record["selected_k"]})

oof_summary_df = pd.DataFrame(oof_summary_rows)
per_fold_metrics_df = pd.DataFrame(per_fold_metric_rows)
print("POOLED NESTED GROUPED OOF METRICS (ROC-AUC/EER ARE SUPPLEMENTARY ACROSS FOLD-SPECIFIC CALIBRATORS)")
print(oof_summary_df.to_string(index=False, float_format=lambda value: f"{value:.6f}"))
print("PER-OUTER-FOLD METRICS")
print(per_fold_metrics_df.to_string(index=False, float_format=lambda value: f"{value:.6f}"))

outer_stats_rows = []
for method, display_name in method_display_names.items():
    method_rows = per_fold_metrics_df[per_fold_metrics_df["method"] == method]
    for metric in metric_names:
        outer_stats_rows.append({"method": method, "Method": display_name, "metric": metric,
                                 "mean": float(method_rows[metric].mean()), "std": float(method_rows[metric].std(ddof=0))})
outer_stats_df = pd.DataFrame(outer_stats_rows)
# Preserve columns used by existing loss-comparison visualization.
per_fold_df = pd.DataFrame([{"outer_fold": record["outer_fold"], "selected_lambda": record["selected_lambda"],
    "selected_k": record["selected_k"], **{method + "_loss": record["metrics_by_method"][method]["log_loss"] for method in method_keys}}
    for record in nested_fold_records])
print("MEAN AND STANDARD DEVIATION ACROSS OUTER FOLDS (all six metrics)")
print(outer_stats_df.to_string(index=False, float_format=lambda value: f"{value:.6f}"))


## Section 7: Component-Level Diagnostics on Nested OOF Predictions

Evaluates how each calibration method performs on held-out recordings across the four speech components:
- `genuine`: Human speech ($y = 0$, 240 recordings)
- `melo`: MeloTTS ($y = 1$, 240 recordings)
- `vits`: VITS ($y = 1$, 240 recordings)
- `xtts`: XTTS ($y = 1$, 240 recordings)

In [ ]:
component_oof_rows = []

for comp_name in ['genuine', 'melo', 'vits', 'xtts']:
    c_mask = cal_df['component'] == comp_name
    c_y = y_true[c_mask]
    rec = {
        "component": comp_name,
        "count": int(np.sum(c_mask)),
        "label": int(c_y[0]),
    }
    for m_key in method_display_names.keys():
        p_sub = nested_oof_preds[m_key][c_mask]
        rec[f"{m_key}_loss"] = compute_bce_loss(c_y, p_sub)
        rec[f"{m_key}_brier"] = compute_brier_score(c_y, p_sub)
    component_oof_rows.append(rec)

comp_oof_df = pd.DataFrame(component_oof_rows)
print("\n" + "=" * 95)
print("  COMPONENT-LEVEL NESTED OUT-OF-FOLD LOG LOSS DIAGNOSTICS")
print("=" * 95)
comp_display_cols = [
    'component', 'label', 'count',
    'raw_loss', 'temperature_loss', 'platt_loss', 'isotonic_loss', 'regularized_isotonic_loss', 'smoothed_isotonic_loss'
]
print(comp_oof_df[comp_display_cols].to_string(index=False, justify='center', float_format=lambda x: f"{x:10.5f}"))
print("=" * 95)

## Section 8: Final Model Fitting on Complete Calibration Set (Reference / Diagnostics)

After completing nested out-of-fold evaluation, we fit all five calibration methods on the complete 960 calibration recordings for diagnostic comparison and artifact export.

> [!NOTE]
> In accordance with scientific integrity guidelines:
> - These full-data fitted parameters are recorded separately from the nested OOF performance.
> - These are reference fits only; no full-data in-sample metric is used to select or declare a winning calibration method.

In [ ]:
print("Fitting reference calibration models on complete calibration dataset (N=960)...")

# 1. Full Temperature
full_temperature = fit_temperature(y_true.tolist(), s_raw.tolist())
full_p_temp = np.array(apply_temperature(s_raw.tolist(), full_temperature))

# 2. Full Platt
full_platt_a, full_platt_b = fit_platt_model(s_raw, y_true)
full_p_platt = apply_platt_model(s_raw, full_platt_a, full_platt_b)

# 3. Full Isotonic
full_iso_model = IsotonicRegression(y_min=0.0, y_max=1.0, out_of_bounds='clip')
full_iso_model.fit(s_raw, y_true)
full_p_iso = np.clip(full_iso_model.predict(s_raw), DEFAULT_EPSILON, 1.0 - DEFAULT_EPSILON)

# 4. Full Regularized Isotonic (using median of outer-selected lambdas: lambda = 0.40)
outer_lambdas = [rec['selected_lambda'] for rec in nested_fold_records]
reference_lambda = float(np.median(outer_lambdas))
full_p_reg_iso = (1.0 - reference_lambda) * full_p_iso + reference_lambda * full_p_temp

# 5. Full Smoothed Isotonic (using median of outer-selected K: K = 10)
outer_ks = [rec['selected_k'] for rec in nested_fold_records]
reference_k = int(np.median(outer_ks))
full_pchip_model = fit_smoothed_pchip(s_raw, full_iso_model, k_knots=reference_k)
full_p_smooth = np.clip(full_pchip_model(s_raw), DEFAULT_EPSILON, 1.0 - DEFAULT_EPSILON)

print(f"  Full Fitted Temperature : T = {full_temperature:.8f}")
print(f"  Full Fitted Platt       : a = {full_platt_a:.8f}, b = {full_platt_b:.8f}")
print(f"  Full Fitted Isotonic    : {len(full_iso_model.X_thresholds_)} thresholds")
print(f"  Full Reg Isotonic       : lambda = {reference_lambda:.2f} (outer fold median)")
print(f"  Full Smoothed Isotonic  : K = {reference_k} knots (outer fold median)")

## Section 9: Visualizations & Reliability Diagrams

Generates publication-quality diagnostic plots for the nested cross-validation experiment:
1. `plots/nested_oof_reliability_comparison.png`: Direct overlay of out-of-fold calibration curves across all five methods.
2. `plots/outer_fold_loss_comparison.png`: Comparison of Log Loss across all 4 outer validation folds.
3. `plots/inner_cv_selection_curves.png`: Inner CV hyperparameter selection curves across outer folds.
4. `plots/reliability_*.png`: Individual reliability diagrams with bin counts and error bars.

In [ ]:
# 1. Figure: Nested OOF Reliability Comparison
fig_comp, ax_comp = plt.subplots(figsize=(7.5, 6.5))
ax_comp.plot([0, 1], [0, 1], 'k--', linewidth=1.5, label='Perfect Calibration (y = x)')

plot_styles = {
    "raw": ("Raw Baseline", "tab:gray", ":", "^"),
    "temperature": ("Temperature Scaling", "tab:blue", "-", "o"),
    "platt": ("Platt Scaling", "tab:green", "--", "s"),
    "isotonic": ("Standard Isotonic", "tab:orange", "-.", "v"),
    "regularized_isotonic": ("Regularized Isotonic", "tab:purple", "-", "D"),
    "smoothed_isotonic": ("Smoothed Isotonic", "teal", "-", "P"),
}

all_reliability_rows = []

for m_key, (m_label, col, ls, mk) in plot_styles.items():
    bw, bm = oof_bins_data[m_key]
    confs = [b['confidence'] for b in bw if not b['is_empty']]
    accs = [b['accuracy'] for b in bw if not b['is_empty']]
    ax_comp.plot(confs, accs, marker=mk, linestyle=ls, color=col, linewidth=1.8, markersize=6, label=m_label)
    
    for b in bw:
        all_reliability_rows.append({"method": m_key, "evaluation": "nested_oof", **b})
    for b in bm:
        all_reliability_rows.append({"method": m_key, "evaluation": "nested_oof", **b})

ax_comp.set_xlabel('Mean Predicted Probability', fontsize=11)
ax_comp.set_ylabel('Empirical Accuracy', fontsize=11)
ax_comp.set_title('Nested Out-of-Fold Reliability Comparison (N=960)', fontsize=12, pad=10)
ax_comp.set_xlim([-0.02, 1.02])
ax_comp.set_ylim([-0.02, 1.02])
ax_comp.legend(loc='upper left', fontsize=9, frameon=True)
ax_comp.grid(True, linestyle=':', alpha=0.6)
fig_comp.tight_layout()

# 2. Figure: Outer Fold Loss Comparison
fig_folds, ax_folds = plt.subplots(figsize=(9, 5))
methods_keys = ['raw', 'temperature', 'platt', 'isotonic', 'regularized_isotonic', 'smoothed_isotonic']
labels_keys = ['Raw', 'Temp', 'Platt', 'Isotonic', 'Reg Iso', 'Smooth Iso']
x_indices = np.arange(len(methods_keys))
bar_width = 0.2

for f_idx in range(4):
    row = per_fold_df.iloc[f_idx]
    f_losses = [
        row['raw_loss'], row['temperature_loss'], row['platt_loss'],
        row['isotonic_loss'], row['reg_isotonic_loss'], row['smoothed_isotonic_loss']
    ]
    ax_folds.bar(x_indices + (f_idx - 1.5) * bar_width, f_losses, width=bar_width, label=f'Outer Fold {f_idx + 1}')

ax_folds.set_xticks(x_indices)
ax_folds.set_xticklabels(labels_keys, fontsize=10)
ax_folds.set_ylabel('Log Loss (BCE)', fontsize=11)
ax_folds.set_title('Outer Fold Log Loss Comparison across Calibration Methods', fontsize=12)
ax_folds.legend(loc='upper right', fontsize=9)
ax_folds.grid(True, linestyle=':', alpha=0.5, axis='y')
fig_folds.tight_layout()

# 3. Figure: Inner CV Selection Curves
fig_inner, (ax_in1, ax_in2) = plt.subplots(1, 2, figsize=(13, 5))
for curve_rec in inner_cv_selection_curves:
    f_num = curve_rec['outer_fold']
    lams = list(curve_rec['reg_curve'].keys())
    l_vals = [curve_rec['reg_curve'][l] for l in lams]
    ax_in1.plot(lams, l_vals, 'o-', label=f"Fold {f_num} (sel={curve_rec['selected_lambda']:.2f})")
    
    ks = list(curve_rec['smooth_curve'].keys())
    k_vals = [curve_rec['smooth_curve'][k] for k in ks]
    ax_in2.plot(ks, k_vals, 's-', label=f"Fold {f_num} (sel={curve_rec['selected_k']})")

ax_in1.set_xlabel('Shrinkage Parameter (lambda)', fontsize=11)
ax_in1.set_ylabel('Inner CV Log Loss', fontsize=11)
ax_in1.set_title('Inner CV: Regularized Isotonic lambda Selection', fontsize=12)
ax_in1.legend(fontsize=9)
ax_in1.grid(True, linestyle=':', alpha=0.5)

ax_in2.set_xlabel('Quantile Knots Count (K)', fontsize=11)
ax_in2.set_ylabel('Inner CV Log Loss', fontsize=11)
ax_in2.set_title('Inner CV: Smoothed Isotonic K Selection', fontsize=12)
ax_in2.legend(fontsize=9)
ax_in2.grid(True, linestyle=':', alpha=0.5)
fig_inner.tight_layout()

# 4. Individual Figures for Each Method
individual_figs = {}
for m_key, (m_label, col, _, mk) in list(plot_styles.items())[1:]:
    fig_ind, ax_ind = plt.subplots(figsize=(6, 5.5))
    ax_ind.plot([0, 1], [0, 1], 'k--', label='Perfect calibration')
    bw, _ = oof_bins_data[m_key]
    confs = [b['confidence'] for b in bw if not b['is_empty']]
    accs = [b['accuracy'] for b in bw if not b['is_empty']]
    ax_ind.plot(confs, accs, marker=mk, color=col, linewidth=2, markersize=7, label=f"OOF {m_label}")
    for b in bw:
        if not b['is_empty']:
            ax_ind.annotate(f"n={b['count']}", (b['confidence'], b['accuracy']), textcoords="offset points", xytext=(0, 7), ha='center', fontsize=8)
    ax_ind.set_xlabel('Mean Predicted Probability', fontsize=11)
    ax_ind.set_ylabel('Empirical Accuracy', fontsize=11)
    ax_ind.set_title(f'Nested OOF Reliability: {m_label}', fontsize=12)
    ax_ind.set_xlim([-0.02, 1.02])
    ax_ind.set_ylim([-0.02, 1.02])
    ax_ind.legend(loc='upper left')
    ax_ind.grid(True, linestyle=':', alpha=0.6)
    fig_ind.tight_layout()
    individual_figs[m_key] = fig_ind

print("Diagnostic visualization figures prepared.")

## Section 10: Export Artifacts & Authoritative Evidence Trail

Creates a new uniquely timestamped run directory under `runs/r3/r3_n1_nested_calibration_<timestamp>/` and writes:
1. `nested_cv_results.json`: Comprehensive structured summary of outer and inner CV splits, per-fold parameters, out-of-fold metrics, and full-fit reference models.
2. `nested_oof_predictions.parquet`: 960 recordings containing `raw_score` and out-of-fold predictions for all five methods.
3. `nested_oof_predictions.csv`: CSV version for inspection.
4. `nested_reliability_data.csv`: Complete bin-level tables for both equal-width and equal-mass binning across methods.
5. `plots/*.png`: All diagnostic plots.
6. `evidence.json`: Audit trail recording input and output SHA-256 digests and boundary verifications.

In [ ]:
RUN_TIMESTAMP = time.strftime('%Y%m%dT%H%M%SZ', time.gmtime())
RUN_ID = f"r3_n1_nested_calibration_{RUN_TIMESTAMP}"
OUTPUT_DIR = ROOT / 'runs/r3' / RUN_ID

assert not OUTPUT_DIR.exists(), f"Output directory already exists: {OUTPUT_DIR}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
PLOTS_DIR = OUTPUT_DIR / 'plots'
PLOTS_DIR.mkdir(parents=True, exist_ok=False)
print(f"[Run Directory Created] {OUTPUT_DIR}")

# 1. Save Plots
fig_comp.savefig(PLOTS_DIR / 'nested_oof_reliability_comparison.png', dpi=200)
plt.close(fig_comp)

fig_folds.savefig(PLOTS_DIR / 'outer_fold_loss_comparison.png', dpi=200)
plt.close(fig_folds)

fig_inner.savefig(PLOTS_DIR / 'inner_cv_selection_curves.png', dpi=200)
plt.close(fig_inner)

for m_key, f_obj in individual_figs.items():
    f_obj.savefig(PLOTS_DIR / f'reliability_{m_key}.png', dpi=200)
    plt.close(f_obj)

# 2. Save Reliability Data CSV
reliability_df = pd.DataFrame(all_reliability_rows)
reliability_df.to_csv(OUTPUT_DIR / 'nested_reliability_data.csv', index=False)

# 3. Save Nested OOF Predictions (Parquet & CSV)
export_cols = [
    'recording_id', 'group_id', 'label', 'component', 'validation_role', 'chunk_count', 'raw_score'
]
oof_df = cal_df[export_cols].copy()
oof_df['outer_fold'] = outer_fold_assignment
assert len(oof_df) == len(cal_df) and oof_df['recording_id'].is_unique
assert np.all(oof_assignment_counts == 1) and oof_df['outer_fold'].between(1, 4).all()
oof_df['oof_score_temperature'] = nested_oof_preds['temperature']
oof_df['oof_score_platt'] = nested_oof_preds['platt']
oof_df['oof_score_isotonic'] = nested_oof_preds['isotonic']
oof_df['oof_score_reg_isotonic'] = nested_oof_preds['regularized_isotonic']
oof_df['oof_score_smoothed_isotonic'] = nested_oof_preds['smoothed_isotonic']

pq.write_table(pa.Table.from_pandas(oof_df, preserve_index=False), OUTPUT_DIR / 'nested_oof_predictions.parquet')
oof_df.to_csv(OUTPUT_DIR / 'nested_oof_predictions.csv', index=False)

# Reload both exports and verify row count, required fields, IDs, fold assignments, and scores.
parquet_reload = pq.read_table(OUTPUT_DIR / 'nested_oof_predictions.parquet').to_pandas()
csv_reload = pd.read_csv(OUTPUT_DIR / 'nested_oof_predictions.csv')
expected_export_columns = list(oof_df.columns)
assert len(parquet_reload) == len(csv_reload) == len(cal_df), "Export row count mismatch"
assert parquet_reload.columns.tolist() == expected_export_columns and csv_reload.columns.tolist() == expected_export_columns
for reloaded in (parquet_reload, csv_reload):
    assert reloaded['recording_id'].notna().all() and reloaded['recording_id'].is_unique
    assert reloaded['outer_fold'].notna().all() and reloaded['outer_fold'].between(1, 4).all()
    score_cols = [column for column in expected_export_columns if column.startswith('oof_score_')]
    score_values = reloaded[score_cols].to_numpy(dtype=np.float64)
    assert np.all(np.isfinite(score_values)) and np.all((score_values >= 0.0) & (score_values <= 1.0))
    assert reloaded['recording_id'].astype(str).tolist() == oof_df['recording_id'].astype(str).tolist()
    assert reloaded['outer_fold'].astype(int).tolist() == outer_fold_assignment.tolist()
    np.testing.assert_allclose(score_values, oof_df[score_cols].to_numpy(dtype=np.float64), rtol=0.0, atol=1e-12)
assert parquet_reload['group_id'].astype(str).tolist() == oof_df['group_id'].astype(str).tolist()
assert csv_reload['group_id'].astype(str).tolist() == oof_df['group_id'].astype(str).tolist()
for source_column in ['label', 'component', 'validation_role', 'chunk_count']:
    expected_values = oof_df[source_column].astype(str).tolist()
    assert parquet_reload[source_column].astype(str).tolist() == expected_values
    assert csv_reload[source_column].astype(str).tolist() == expected_values
np.testing.assert_allclose(parquet_reload['raw_score'].to_numpy(dtype=np.float64), oof_df['raw_score'].to_numpy(dtype=np.float64), rtol=0.0, atol=1e-12)
np.testing.assert_allclose(csv_reload['raw_score'].to_numpy(dtype=np.float64), oof_df['raw_score'].to_numpy(dtype=np.float64), rtol=0.0, atol=1e-12)

# 4. Save nested_cv_results.json
nested_cv_payload = {
    "task": "R3-N1 Stage 1 Nested Grouped Cross-Validation",
    "run_id": RUN_ID,
    "created_utc": RUN_TIMESTAMP,
    "source_prediction_path": str(CALIBRATION_PRED_FILE.relative_to(ROOT)),
    "source_prediction_sha256": source_pred_sha256,
    "model_architecture": "BiGRU",
    "protocol": "B",
    "dataset": {
        "role": "calibration",
        "total_recordings": len(cal_df),
        "class_counts": {"genuine_0": int(label_counts[0]), "tts_1": int(label_counts[1])},
        "component_counts": comp_counts,
        "speaker_groups_count": unique_groups,
    },
    "nested_cv_structure": {
        "outer_cv": "GroupKFold(n_splits=4, grouped by group_id)",
        "inner_cv": "GroupKFold(n_splits=3, grouped by group_id)",
        "outer_folds_count": 4,
        "inner_folds_count": 3,
        "outer_fold_records": nested_fold_records,
    },
    "pooled_nested_oof_metrics": oof_summary_df.to_dict(orient='records'),
    "per_outer_fold_metrics": per_fold_metrics_df.to_dict(orient='records'),
    "outer_fold_summary_stats_all_metrics": outer_stats_df.to_dict(orient='records'),
    "fold_manifest": fold_manifest,
    "recording_outer_fold_assignments": [{"recording_id": str(rid), "outer_fold": int(fold)} for rid, fold in zip(cal_df['recording_id'], outer_fold_assignment)],
    "oof_assignment_counts": {"all_exactly_once": bool(np.all(oof_assignment_counts == 1)), "minimum": int(oof_assignment_counts.min()), "maximum": int(oof_assignment_counts.max())},
    "outer_fold_summary_stats": outer_stats_df.to_dict(orient='records'),
    "component_oof_diagnostics": comp_oof_df.to_dict(orient='records'),
    "final_full_calibration_fit": {
        "temperature_T": float(full_temperature),
        "platt_a": float(full_platt_a),
        "platt_b": float(full_platt_b),
        "isotonic_thresholds_count": len(full_iso_model.X_thresholds_),
        "regularized_isotonic_reference_lambda": float(reference_lambda),
        "smoothed_isotonic_reference_k": int(reference_k),
    },
}

with (OUTPUT_DIR / 'nested_cv_results.json').open('w', encoding='utf-8') as f:
    json.dump(nested_cv_payload, f, indent=2)

# 5. Compute Output Hashes for Evidence Record
output_files = [
    'nested_cv_results.json',
    'nested_oof_predictions.parquet',
    'nested_oof_predictions.csv',
    'nested_reliability_data.csv',
    'plots/nested_oof_reliability_comparison.png',
    'plots/outer_fold_loss_comparison.png',
    'plots/inner_cv_selection_curves.png',
    'plots/reliability_temperature.png',
    'plots/reliability_platt.png',
    'plots/reliability_isotonic.png',
    'plots/reliability_regularized_isotonic.png',
    'plots/reliability_smoothed_isotonic.png',
]

output_sha256 = {}
for rel_path in output_files:
    f_path = OUTPUT_DIR / rel_path
    output_sha256[rel_path] = compute_sha256(f_path)

# 6. Save evidence.json
evidence_payload = {
    "task": "R3-N1 Stage 1 Nested CV",
    "run_id": RUN_ID,
    "created_utc": RUN_TIMESTAMP,
    "status": "nested_grouped_calibration_complete",
    "source_file": {
        "path": str(CALIBRATION_PRED_FILE.relative_to(ROOT)),
        "sha256": source_pred_sha256,
        "bytes": source_pred_size,
    },
    "output_sha256": output_sha256,
    "dataset_counts": {
        "total": len(cal_df),
        "genuine": int(label_counts[0]),
        "tts": int(label_counts[1]),
        "components": comp_counts,
        "speaker_groups": unique_groups,
    },
    "outer_cv": {
        "splits": 4,
        "group_variable": "group_id",
        "held_out_groups_per_fold": 3,
        "held_out_recordings_per_fold": 240,
    },
    "inner_cv": {
        "splits": 3,
        "group_variable": "group_id",
        "held_out_groups_per_fold": 3,
        "held_out_recordings_per_fold": 240,
    },
    "fold_manifest": fold_manifest,
    "recording_outer_fold_assignments": [{"recording_id": str(rid), "outer_fold": int(fold)} for rid, fold in zip(cal_df['recording_id'], outer_fold_assignment)],
    "oof_assignment_counts": {"all_exactly_once": bool(np.all(oof_assignment_counts == 1)), "minimum": int(oof_assignment_counts.min()), "maximum": int(oof_assignment_counts.max())},
    "per_fold_selected_hyperparameters": [
        {"fold": r["outer_fold"], "lambda": r["selected_lambda"], "K": r["selected_k"]}
        for r in nested_fold_records
    ],
    "scientific_boundaries": {
        "validation_role_used": "calibration",
        "operating_set_accessed": False,
        "test_partition_accessed": False,
        "chatterbox_accessed": False,
        "model_retrained": False,
        "original_predictions_modified": False,
        "nested_speaker_grouped_cv": True,
        "zero_outer_fold_leakage": bool(all(set(row["outer_train_groups"]).isdisjoint(row["outer_validation_groups"]) for row in fold_manifest)),
        "oof_predictions_complete": bool(np.all(oof_assignment_counts == 1) and all(np.all(np.isfinite(values)) for values in nested_oof_preds.values())),
    },
}

with (OUTPUT_DIR / 'evidence.json').open('w', encoding='utf-8') as f:
    json.dump(evidence_payload, f, indent=2)

assert len(oof_df) == len(cal_df) and len(parquet_reload) == len(csv_reload) == len(cal_df)
assert all(len(row["inner_folds"]) == 3 for row in fold_manifest) and len(fold_manifest) == 4
print("[Artifacts reloaded and validated; fold manifest and assignment evidence exported]")

## Section 11: Final Integrity Auditing & Reproducibility Verification

Audits all exported artifacts, validates that the source predictions file remained byte-for-byte unmodified, checks that all output files exist with matching SHA-256 digests, and summarizes the nested cross-validation evaluation.

In [ ]:
# 1. Verify Source Prediction File Hash Unchanged
current_source_sha256 = compute_sha256(CALIBRATION_PRED_FILE)
assert current_source_sha256 == source_pred_sha256, (
    f"CRITICAL ERROR: Original prediction file was modified during calibration!\n"
    f"  Before: {source_pred_sha256}\n"
    f"  After : {current_source_sha256}"
)
print("[Source Integrity Verified] Original predictions file is byte-for-byte unchanged.")

# 2. Verify Output Files Exist and Match Recorded Hashes
for filename, recorded_digest in output_sha256.items():
    file_path = OUTPUT_DIR / filename
    assert file_path.is_file(), f"Output file missing: {file_path}"
    actual_digest = compute_sha256(file_path)
    assert actual_digest == recorded_digest, f"Digest mismatch on {filename}: {actual_digest} != {recorded_digest}"
    print(f"  [OK] {filename:<40} ({file_path.stat().st_size:>8} bytes) sha={actual_digest[:16]}...")

# 3. Final Summary Banner
print("\n" + "=" * 95)
print(f"{'R3-N1 STAGE 1: NESTED GROUPED CROSS-VALIDATION COMPLETE':^95}")
print("=" * 95)
print(f"Run ID             : {RUN_ID}")
print(f"Artifacts Location : {OUTPUT_DIR.relative_to(ROOT)}")
print(f"Methods Evaluated  : 5 methods via 4-fold Outer / 3-fold Inner Nested CV")
print(f"Recordings Fitted  : {len(cal_df)} (Genuine=240, TTS=720, 12 Speaker Groups)")
print("Metrics            : six metrics pooled and by outer fold; pooled ROC-AUC/EER supplementary")
print(f"Boundaries         : Calibration-only | Strict zero leakage | Operating & Test unaccessed")
print("=" * 95)
print("Nested grouped OOF estimates and export integrity checks completed.")